# TalentIQ - Preprocessing and Feature Engineering 

This notebook explains everything that happens in `preprocessing.py` and `feature_engineering.py`. The goal is to understand not just what the code does, but why each step exists.

## What is preprocessing?

Raw data is never perfect. It can have missing values, duplicate rows, wrong data types, and extreme outliers. If we feed that raw data directly into a machine learning model, the model will learn the wrong things and perform poorly.

Preprocessing is the step where we clean and prepare the data so the model can actually learn from it. In this project, the preprocessing pipeline runs in this order: load the raw CSV, validate that required columns exist, fill any missing values, remove duplicates, handle outliers, and save the cleaned data as a new file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load raw data to demonstrate each step
df_raw = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print('Raw shape:', df_raw.shape)

## Step 1 - Loading the data

The `load_data()` function reads the mode from `config.yaml`. In sample mode it loads 500 rows for quick testing, and in full mode it loads all 1470 rows for the final run.

After loading, a few things happen immediately. Four columns that carry no useful information are dropped: `EmployeeNumber` is just an ID, `EmployeeCount` is always 1, `Over18` is always Y, and `StandardHours` is always 80. Keeping them would only add noise.

The `Attrition` column gets converted from Yes/No text to 1/0 since models only understand numbers. Some columns like `JobSatisfaction` and `Education` are stored as integers but the numbers carry specific meaning (Low, Medium, High, Very High). These get converted to their text labels here so the ordinal encoder can later learn the correct order.

In [ ]:
# Demonstrate what load_data() does step by step

df = df_raw.copy()

# Drop useless constant columns
drop_cols = ['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print('After dropping useless columns:', df.shape)

# Convert target to numeric
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
print('Attrition value counts after conversion:')
print(df['Attrition'].value_counts())

# Convert integer ordinal columns to string labels
col_maps = {
    'Education': {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'},
    'JobSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'WorkLifeBalance': {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'},
}
for col, mapping in col_maps.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

print('\nJobSatisfaction sample values after mapping:')
print(df['JobSatisfaction'].value_counts())

## Step 2 - Validating the data

Before processing anything, `validate_data()` checks whether the data is actually usable. This is a safety check — if someone passes in the wrong file, the pipeline should fail loudly with a clear error rather than silently producing wrong results.

It checks three things: whether the dataframe is empty, whether the target column Attrition exists, and whether all required feature columns are present. The list of required columns comes from `features.yaml` so if we add a new feature there, the validation automatically picks it up.

In [ ]:
# Show what validate_data() checks

required_numerical = [
    'Age', 'MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany',
    'YearsWithCurrManager', 'NumCompaniesWorked', 'YearsSinceLastPromotion'
]
required_categorical = ['BusinessTravel', 'Department', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
target = 'Attrition'

all_required = required_numerical + required_categorical + [target]
missing = [col for col in all_required if col not in df.columns]

if missing:
    print('Missing columns:', missing)
else:
    print('Validation passed. All required columns are present.')
print('DataFrame is empty:', df.empty)

## Step 3 - Handling missing values

Even though this dataset has no missing values, the code is written defensively because real world data often has gaps and we want this pipeline to work on any dataset.

For numerical columns like Age and MonthlyIncome, missing values are filled with the median rather than the mean. The reason is that the mean gets pulled by outliers — if one employee earns 500,000 while everyone else earns 5,000, the mean is misleading. The median gives a more accurate middle value.

For categorical columns like Department and JobRole, missing values are filled with the mode, which is the most common value. You cannot average text categories, so the most frequent one is the safest guess.

In [ ]:
# Demonstrate the logic of handle_missing_values()

# Artificially introduce missing values to show the logic
df_demo = df.copy()
df_demo.loc[0:10, 'MonthlyIncome'] = np.nan
df_demo.loc[5:15, 'Department'] = np.nan

print('Missing values before:')
print(df_demo[['MonthlyIncome', 'Department']].isnull().sum())

# Fix numerical with median
df_demo['MonthlyIncome'] = df_demo['MonthlyIncome'].fillna(df_demo['MonthlyIncome'].median())

# Fix categorical with mode
df_demo['Department'] = df_demo['Department'].fillna(df_demo['Department'].mode()[0])

print('\nMissing values after:')
print(df_demo[['MonthlyIncome', 'Department']].isnull().sum())
print('\nMedian MonthlyIncome used:', df['MonthlyIncome'].median())
print('Mode Department used:', df['Department'].mode()[0])

## Step 4 - Removing duplicates

If the same employee row appears twice, the model sees the same example twice. It is like reading the same flashcard twice in a row — it does not improve learning and can actually bias the model toward whichever rows appear more often. `remove_duplicates()` drops any row where every column value is identical to another row and logs how many were removed.

In [ ]:
before = len(df)
df_dedup = df.drop_duplicates()
after = len(df_dedup)
print(f'Rows before: {before}')
print(f'Rows after: {after}')
print(f'Duplicates removed: {before - after}')

## Step 5 - Handling outliers with IQR

An outlier is a value that is extremely different from everything else in that column. If most employees earn between 2,000 and 15,000 per month but one earns 100,000, that single row can distort what the model learns.

The IQR method (Interquartile Range) measures the spread of the middle 50% of the data. Q1 is the value at the 25th percentile and Q3 is at the 75th. IQR is Q3 minus Q1. The lower bound is Q1 minus 1.5 times the IQR, and the upper bound is Q3 plus 1.5 times the IQR. Any value outside these bounds is considered an outlier.

Rather than deleting those rows, we clip them — a value of 100,000 that exceeds the upper bound of 25,000 gets replaced with 25,000. The row stays but the extreme value is brought back into a reasonable range. This is applied to MonthlyIncome, TotalWorkingYears, YearsAtCompany, and DistanceFromHome.

In [ ]:
# Demonstrate IQR outlier handling step by step

col = 'MonthlyIncome'
data = df[col].copy()

q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f'Column: {col}')
print(f'Q1: {q1:.0f}')
print(f'Q3: {q3:.0f}')
print(f'IQR: {iqr:.0f}')
print(f'Lower bound: {lower:.0f}')
print(f'Upper bound: {upper:.0f}')
print(f'Values above upper bound: {(data > upper).sum()}')
print(f'Max before clipping: {data.max():.0f}')

clipped = np.clip(data, lower, upper)
print(f'Max after clipping: {clipped.max():.0f}')

In [ ]:
# Visual comparison before and after clipping
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(data, bins=30, color='tomato')
axes[0].set_title('MonthlyIncome - Before Clipping')
axes[0].set_xlabel('Income')

axes[1].hist(clipped, bins=30, color='steelblue')
axes[1].set_title('MonthlyIncome - After Clipping')
axes[1].set_xlabel('Income')

plt.tight_layout()
plt.show()
print('The right tail is compressed. Extreme values are brought back to the boundary.')

## Step 6 - Saving the processed data

After all the cleaning steps, the processed dataframe is saved to a CSV file. The path comes from `config.yaml`: full mode saves to `data/processed/processed_full.csv` and sample mode saves to `data/processed/processed_sample.csv`.

Saving at this point means we do not have to re-run preprocessing every time we train. It also creates a checkpoint — if something breaks in training, we can inspect the processed file to see whether preprocessing was correct.

In [ ]:
# Load the processed data to demonstrate feature engineering
try:
    df_proc = pd.read_csv('../data/processed/processed_full.csv')
    print('Loaded processed data:', df_proc.shape)
except FileNotFoundError:
    # If not yet saved, use raw with basic mapping applied
    df_proc = df.copy()
    print('Using raw data for demonstration:', df_proc.shape)

## Feature 1 - IncomePerYear

Formula: `MonthlyIncome / (YearsAtCompany + 1)`

An employee who has been at the company for 10 years but still earns a low salary might feel undervalued and is more likely to leave than someone earning the same salary after only one year. This feature captures that ratio directly. A low IncomePerYear signals that the person is not being compensated in proportion to their tenure. We add 1 to YearsAtCompany to avoid dividing by zero for new employees.

In [ ]:
df_proc['IncomePerYear'] = df_proc['MonthlyIncome'] / (df_proc['YearsAtCompany'] + 1)

print('IncomePerYear statistics:')
print(df_proc['IncomePerYear'].describe().round(2))

# Show average IncomePerYear for those who left vs stayed
print('\nAverage IncomePerYear by Attrition:')
print(df_proc.groupby('Attrition')['IncomePerYear'].mean().round(2))

## Feature 2 - SatisfactionScore

Formula: average of `JobSatisfaction`, `EnvironmentSatisfaction`, `RelationshipSatisfaction`, and `WorkLifeBalance`

Each of these four columns measures a different aspect of how satisfied an employee is. Instead of treating them as four separate signals, we combine them into one overall satisfaction score. The text labels are first mapped back to numbers (Low=1, Medium=2, High=3, Very High=4) and then averaged across all four columns. A lower average means the employee is generally dissatisfied across multiple areas, which is a strong signal for attrition.

In [ ]:
sat_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4,
           'Bad': 1, 'Good': 2, 'Better': 3, 'Best': 4}

satisfaction_cols = [c for c in ['JobSatisfaction', 'EnvironmentSatisfaction',
                                  'RelationshipSatisfaction', 'WorkLifeBalance']
                     if c in df_proc.columns]

sat_df = df_proc[satisfaction_cols].replace(sat_map).apply(pd.to_numeric, errors='coerce')
df_proc['SatisfactionScore'] = sat_df.mean(axis=1)

print('SatisfactionScore statistics:')
print(df_proc['SatisfactionScore'].describe().round(2))

print('\nAverage SatisfactionScore by Attrition:')
print(df_proc.groupby('Attrition')['SatisfactionScore'].mean().round(3))

## Feature 3 - LoyaltyIndex

Formula: `YearsWithCurrManager / (TotalWorkingYears + 1)`

If an employee has spent a large portion of their total career under their current manager, it suggests a strong relationship. A low LoyaltyIndex means they recently switched managers or have a long career history but very little time with the current one. This can indicate instability or dissatisfaction following a management change.

In [ ]:
df_proc['LoyaltyIndex'] = df_proc['YearsWithCurrManager'] / (df_proc['TotalWorkingYears'] + 1)

print('LoyaltyIndex statistics:')
print(df_proc['LoyaltyIndex'].describe().round(3))

print('\nAverage LoyaltyIndex by Attrition:')
print(df_proc.groupby('Attrition')['LoyaltyIndex'].mean().round(3))

## Feature 4 - WorkloadScore

Formula: `TrainingTimesLastYear * JobInvolvement (as number)`

An employee who is highly involved in their job and also attends many training sessions is either very engaged or very overloaded. This score combines the intensity of their involvement with how much training they are doing. When paired with other features it helps the model understand burnout risk.

In [ ]:
inv_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4}
if 'JobInvolvement' in df_proc.columns:
    job_inv = df_proc['JobInvolvement'].map(inv_map).fillna(2)
else:
    job_inv = 2

df_proc['WorkloadScore'] = df_proc['TrainingTimesLastYear'] * job_inv

print('WorkloadScore statistics:')
print(df_proc['WorkloadScore'].describe().round(2))

## Feature 5 - CareerGrowthRate

Formula: `(YearsAtCompany - YearsSinceLastPromotion) / (YearsAtCompany + 1)`

This measures how recently an employee was promoted relative to how long they have been at the company. Someone who has been here 8 years with their last promotion 7 years ago gets a low ratio — they are stagnating. A higher ratio means they were promoted more recently relative to their tenure, which suggests active career growth and lower risk of leaving.

In [ ]:
df_proc['CareerGrowthRate'] = (
    (df_proc['YearsAtCompany'] - df_proc['YearsSinceLastPromotion']) /
    (df_proc['YearsAtCompany'] + 1)
)

print('CareerGrowthRate statistics:')
print(df_proc['CareerGrowthRate'].describe().round(3))

print('\nAverage CareerGrowthRate by Attrition:')
print(df_proc.groupby('Attrition')['CareerGrowthRate'].mean().round(3))

## Feature 6 - OverTimeRisk

Formula: `1 if OverTime == Yes, else 0`

The EDA showed that OverTime is the single strongest predictor of attrition — employees who do overtime leave at a much higher rate. The `OverTime` column already exists as text (Yes/No). We create a binary version called `OverTimeRisk` so the model can use it as a plain number. This is kept as a separate 0/1 column rather than being folded into the one-hot encoding that happens later.

In [ ]:
if 'OverTime' in df_proc.columns:
    df_proc['OverTimeRisk'] = (df_proc['OverTime'] == 'Yes').astype(int)
else:
    df_proc['OverTimeRisk'] = 0

print('OverTimeRisk distribution:')
print(df_proc['OverTimeRisk'].value_counts())

print('\nAttrition rate by OverTimeRisk:')
print(df_proc.groupby('OverTimeRisk')['Attrition'].mean().round(3))

## Feature 7 - TenureStabilityIndex

Formula: `TotalWorkingYears / (NumCompaniesWorked + 1)`

This measures how stable an employee's career history has been. Someone with 15 total working years across 8 different companies has a low stability index — they are a job hopper and historically more likely to leave again. Someone with 15 years across only 2 companies has a high stability index, suggesting they tend to stay long term.

In [ ]:
df_proc['TenureStabilityIndex'] = df_proc['TotalWorkingYears'] / (df_proc['NumCompaniesWorked'] + 1)

print('TenureStabilityIndex statistics:')
print(df_proc['TenureStabilityIndex'].describe().round(2))

print('\nAverage TenureStabilityIndex by Attrition:')
print(df_proc.groupby('Attrition')['TenureStabilityIndex'].mean().round(3))

## Summary of all engineered features

The seven features added are IncomePerYear, SatisfactionScore, LoyaltyIndex, WorkloadScore, CareerGrowthRate, OverTimeRisk, and TenureStabilityIndex. The cells below print their statistics and show how each one correlates with the Attrition target.

In [ ]:
engineered = ['IncomePerYear', 'SatisfactionScore', 'LoyaltyIndex',
              'WorkloadScore', 'CareerGrowthRate', 'OverTimeRisk', 'TenureStabilityIndex']

print('All 7 engineered features added to the dataframe:')
print(df_proc[engineered].describe().round(3))

In [ ]:
# Show correlation of each engineered feature with Attrition
corr_with_target = df_proc[engineered + ['Attrition']].corr()['Attrition'].drop('Attrition')
corr_with_target = corr_with_target.sort_values(key=abs, ascending=False)

print('Correlation of engineered features with Attrition:')
print(corr_with_target.round(3))

## What happens next?

After preprocessing and feature engineering, the data goes into the training pipeline where four more transformations happen.

Ordinal encoding is applied to columns like Education and JobSatisfaction where the categories have a meaningful order (Low < Medium < High < Very High). The model needs to understand that Doctor is higher than Bachelor, which one-hot encoding would not capture.

One-hot encoding is used for columns like Department and JobRole where there is no meaningful order between categories. Treating them as numbers would imply that HR is three times greater than Sales, which makes no sense.

Standard scaling is applied to all numerical columns. MonthlyIncome ranges from 1,000 to 20,000 while Age ranges from 18 to 60. Without scaling, models like Logistic Regression would unfairly weight MonthlyIncome more just because the numbers are larger. Scaling brings every column to mean=0 and standard deviation=1 so they are on the same relative scale.

Finally, SMOTE generates synthetic training examples of the minority class to balance the training data before the model trains. All of this happens inside `train.py` using a ColumnTransformer and imblearn. See the Training Pipeline notebook for the full explanation.